# Clase 2 · Embeddings: el significado como vector

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hizocar/analitica-negocios-2/blob/main/docs/notebooks/clase2_embeddings.ipynb)

**Analítica para los Negocios II · USM** · Notebook de la Clase 2 · Se aplica en el **Sprint 2 (domingo 15 de noviembre, 23:59)**.

En la cápsula web usaste embeddings **precalculados** de 400 reclamos. Aquí los vas a calcular **tú** para los ~6.000 textos de tu caso y armarás todas las piezas del Sprint 2:

1. Instalar `sentence-transformers` y cargar + limpiar los datos (**FinTrust** o **MercaSur**).
2. Calcular tu **primer embedding** y comparar similitudes con TF-IDF.
3. Calcular los embeddings de **todo el dataset** (en lotes, midiendo el tiempo) y **guardarlos** con `np.save` (los reutilizarás en la Clase 4).
4. Construir un **buscador semántico** con una función reutilizable.
5. Comparar un **clasificador sobre embeddings vs. TF-IDF** (F1 macro).
6. Dibujar la **curva de aprendizaje con pocos datos** (50 / 200 / 1.000 ejemplos) y hacer la **prueba de paráfrasis**.
7. Dibujar el **mapa 2D** (PCA) coloreado por clase.
8. **Descubrir temas** con K-means y ponerle nombre a cada cluster.

> 💡 Primero haz **Archivo → Guardar una copia en Drive**. Recomendado: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)** (los embeddings se calculan mucho más rápido). **No necesitas clave de API**. Con GPU, todo corre en ~3 minutos.

## 0. Preparación

Instalamos `sentence-transformers` **solo si falta** (Colab normalmente ya lo trae) y elegimos el caso cambiando **una sola línea**.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("sentence_transformers") is None:
    print("Instalando sentence-transformers…")
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-U", "sentence-transformers"], check=True)
print("sentence-transformers listo ✅")

In [ ]:
CASO = "fintrust"  # o "mercasur"

BASE = "https://raw.githubusercontent.com/hizocar/analitica-negocios-2/main/docs/datasets/"
MODELO_EMB = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

import time
import unicodedata
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 140)
SEMILLA = 42

CONFIG = {
    "fintrust": {
        "archivo": "reclamos_fintrust.csv", "texto": "texto", "clase": "categoria",
        "clases": ["Fraude", "Cobro indebido", "App y canales digitales",
                   "Crédito y tasas", "Atención al cliente", "Información y trámites"],
        "sinonimos": {"cobros indebidos": "Cobro indebido"},
    },
    "mercasur": {
        "archivo": "resenas_mercasur.csv", "texto": "texto", "clase": "aspecto",
        "clases": ["Despacho", "Calidad del producto", "Precio",
                   "Atención", "Devoluciones y garantía", "Pago"],
        "sinonimos": {"calidad producto": "Calidad del producto", "pagos": "Pago"},
    },
}
cfg = CONFIG[CASO]
COL_TEXTO, COL_CLASE = cfg["texto"], cfg["clase"]
print(f"Caso: {CASO} · archivo: {cfg['archivo']} · clase: {COL_CLASE}")

## 1. Cargar y limpiar los datos

Usamos la misma limpieza mínima del Sprint 1 como **función reutilizable**: eliminar duplicados exactos y textos vacíos o basura, y normalizar las categorías mal escritas ("FRAUDE", "Credito y tasas", "Pagos"…) a su nombre oficial.

In [ ]:
def sin_tildes(s):
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")

def clave(s):
    # '  Crédito y  TASAS ' -> 'credito y tasas'
    return " ".join(sin_tildes(str(s)).lower().split())

MAPA = {clave(c): c for c in cfg["clases"]}
MAPA.update(cfg["sinonimos"])

def limpiar(df):
    df = df.drop_duplicates().copy()
    df[COL_TEXTO] = df[COL_TEXTO].fillna("").astype(str).str.strip()
    df = df[df[COL_TEXTO].str.split().str.len() >= 3]                      # fuera vacíos, "...", "reclamo"
    df[COL_CLASE] = df[COL_CLASE].map(lambda s: MAPA.get(clave(s)) if pd.notna(s) else None)
    df = df.dropna(subset=[COL_CLASE])
    return df.reset_index(drop=True)

df_raw = pd.read_csv(BASE + cfg["archivo"])
df = limpiar(df_raw)
textos = df[COL_TEXTO].tolist()
y = df[COL_CLASE].to_numpy()
print(f"Filas: {len(df_raw)} -> {len(df)} después de limpiar")
df[COL_CLASE].value_counts()

## 2. Tu primer embedding

Cargamos el modelo **`paraphrase-multilingual-MiniLM-L12-v2`**: una red transformer de 12 capas, entrenada para que frases con el mismo significado (en más de 50 idiomas) queden cerca. La primera vez descarga ~470 MB.

- `model.encode(lista_de_textos)` devuelve una matriz: **una fila por texto, 384 columnas**.
- Con `normalize_embeddings=True` cada vector queda de **largo 1**, así que el **coseno es simplemente el producto punto** (`A @ B.T`).

Repetimos el experimento del Tema 1: frases que significan lo mismo con palabras distintas, y una que comparte palabras pero significa otra cosa.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODELO_EMB)
print("Dispositivo:", model.device)

FRASES = {
    "fintrust": ["Me clonaron la tarjeta",
                 "Aparecen compras que yo no hice",
                 "Unos delincuentes usaron mi plástico",
                 "Quiero cambiar la fecha de pago de mi tarjeta"],
    "mercasur": ["El pedido llegó con una semana de atraso",
                 "El courier todavía no me entrega el paquete",
                 "Se demoraron muchísimo en traerlo",
                 "El pedido llegó y la calidad es excelente"],
}[CASO]

emb = model.encode(FRASES, normalize_embeddings=True)
print("Forma de la matriz:", emb.shape)
print("Primeros 5 números de la frase 1:", emb[0, :5].round(3))

tf = TfidfVectorizer().fit_transform(FRASES)
etiquetas = list("ABCD")
print("\nSimilitud coseno con EMBEDDINGS")
display(pd.DataFrame(emb @ emb.T, index=etiquetas, columns=etiquetas).round(2))
print("Similitud coseno con TF-IDF")
display(pd.DataFrame((tf @ tf.T).toarray(), index=etiquetas, columns=etiquetas).round(2))
for e, f in zip(etiquetas, FRASES):
    print(e, "·", f)

**¿Qué mirar?** Con embeddings, A, B y C (mismo significado, otras palabras) tienen cosenos altos entre sí. Con TF-IDF son ~0, y la frase D (que comparte palabras con A pero significa otra cosa) es la que más se "parece" a A. Ojo: los embeddings tampoco son perfectos; fíjate cuánto se parece D a A según el modelo.

## 3. Embeddings de todo el dataset

Calculamos los embeddings de los ~6.000 textos **en lotes** (`batch_size=64`): el modelo procesa 64 textos a la vez, lo que aprovecha mucho mejor la GPU. Medimos el tiempo, porque en un proyecto real el costo de cómputo importa.

Después los **guardamos** con `np.save`: así no tienes que recalcularlos cada vez, y **los reutilizarás en la Clase 4 (RAG)**. Guardamos también los textos limpios y el nombre del modelo (¡los embeddings de modelos distintos no se pueden mezclar!).

In [ ]:
t0 = time.time()
E = model.encode(textos, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
segundos = time.time() - t0
print(f"Embeddings: {E.shape} · {segundos:.1f} s · {1000 * segundos / len(textos):.1f} ms por texto")

np.save(f"embeddings_{CASO}.npy", E)
df[[COL_TEXTO, COL_CLASE]].to_csv(f"textos_limpios_{CASO}.csv", index=False)
with open(f"embeddings_{CASO}_modelo.txt", "w") as f:
    f.write(MODELO_EMB)
print(f"Guardados: embeddings_{CASO}.npy, textos_limpios_{CASO}.csv y embeddings_{CASO}_modelo.txt")
print("💡 Descárgalos (panel de archivos a la izquierda) o cópialos a tu Drive: los usarás en la Clase 4.")

> Para volver a cargarlos más adelante: `E = np.load(f"embeddings_{CASO}.npy")`. Recuerda que la fila *i* de `E` corresponde a la fila *i* de `textos_limpios_{CASO}.csv`.

## 4. Buscador semántico

La receta: **consulta → vector (mismo modelo) → coseno con todos → top-k**. Como los vectores están normalizados, el coseno con todos es una sola multiplicación `E @ q`.

Armamos una función reutilizable y la comparamos con un buscador por palabras (TF-IDF). Cambia `CONSULTAS` por **tus 5 consultas de negocio** para el Sprint 2.

In [ ]:
def buscar(consulta, k=5):
    # Top-k textos más parecidos a la consulta según embeddings (coseno)
    q = model.encode([consulta], normalize_embeddings=True)[0]
    sims = E @ q
    top = np.argsort(-sims)[:k]
    return pd.DataFrame({"similitud": sims[top].round(3), "clase": y[top], "texto": np.array(textos)[top]})

vectorizador = TfidfVectorizer(sublinear_tf=True)
T = vectorizador.fit_transform(textos)

def buscar_tfidf(consulta, k=5):
    # Mismo buscador, pero por palabras compartidas (TF-IDF)
    sims = (T @ vectorizador.transform([consulta]).T).toarray().ravel()
    top = np.argsort(-sims)[:k]
    return pd.DataFrame({"similitud": sims[top].round(3), "clase": y[top], "texto": np.array(textos)[top]})

CONSULTAS = {
    "fintrust": ["clientes víctimas de phishing", "problemas para entrar a la app",
                 "quiero pagar mi deuda antes de tiempo", "nadie responde mis reclamos",
                 "perdí mi trabajo y no puedo pagar"],
    "mercasur": ["el producto llegó dañado", "me cobraron más de lo que decía la publicación",
                 "no me quieren devolver la plata", "el vendedor fue muy amable",
                 "la página falla al momento de pagar"],
}[CASO]

for c in CONSULTAS:
    print(f"\n🔎 «{c}»")
    display(buscar(c))

Compara con el buscador por palabras en una consulta. ¿Cuántos de los 5 resultados son relevantes en cada caso? Esa cuenta ("precisión en el top-5") es una forma simple y honesta de evaluar un buscador en tu informe.

In [ ]:
consulta = CONSULTAS[0]
print("🧠 Embeddings:"); display(buscar(consulta))
print("🔤 TF-IDF:");     display(buscar_tfidf(consulta))

## 5. Clasificador: embeddings vs. TF-IDF

Usamos los embeddings como **variables** de una regresión logística y los comparamos con el baseline del Sprint 1 (TF-IDF + logística) en el **mismo test**. Importante: el `TfidfVectorizer` se ajusta **solo con train** (si lo ajustas con todo, filtras información del test).

In [ ]:
idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=0.3, stratify=y, random_state=SEMILLA)
textos_np = np.array(textos)

# A) Embeddings + logística
m_emb = LogisticRegression(C=10, max_iter=2000).fit(E[idx_tr], y[idx_tr])
pred_emb = m_emb.predict(E[idx_te])

# B) TF-IDF + logística (baseline Sprint 1)
tfidf = TfidfVectorizer(sublinear_tf=True)
X_tr = tfidf.fit_transform(textos_np[idx_tr])
m_tf = LogisticRegression(C=10, max_iter=2000).fit(X_tr, y[idx_tr])
pred_tf = m_tf.predict(tfidf.transform(textos_np[idx_te]))

resultados = pd.DataFrame({
    "F1 macro (test)": [f1_score(y[idx_te], pred_emb, average="macro"), f1_score(y[idx_te], pred_tf, average="macro")],
    "N.º de variables": [E.shape[1], X_tr.shape[1]],
}, index=["Embeddings + logística", "TF-IDF + logística"]).round(3)
display(resultados)
print(classification_report(y[idx_te], pred_emb, digits=3))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y[idx_te], pred_emb, xticks_rotation=45, ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Embeddings + logística · matriz de confusión (test)")
plt.tight_layout(); plt.show()

## 6. ¿Y con pocos datos? Curva de aprendizaje + prueba de paráfrasis

Etiquetar textos cuesta (alguien tiene que leerlos). La pregunta de negocio: **¿cuántos ejemplos etiquetados necesito y qué método conviene?**

Entrenamos ambos modelos con **50, 200 y 1.000** ejemplos (más 25 como caso extremo), repitiendo con 3 sorteos, y evaluamos en dos conjuntos:

- **Test del caso:** textos del mismo estilo que el entrenamiento.
- **Paráfrasis:** 36 textos escritos a mano **con otras palabras** (6 por clase), que simulan clientes nuevos o un canal nuevo.

In [ ]:
PARAFRASIS = {
    "fintrust": {
        "Fraude": ["Me llegó un correo falso del banco, puse mis datos y me vaciaron la cuenta.",
                   "Hay movimientos en mi cartola que jamás autoricé.",
                   "Unos delincuentes usaron mi plástico para comprar por internet.",
                   "Me hackearon y sacaron plata sin mi permiso.",
                   "Me estafaron por WhatsApp haciéndose pasar por el soporte del banco.",
                   "Perdí mi cartera y alguien ocupó la tarjeta en varias tiendas."],
        "Cobro indebido": ["Me descontaron dos veces el mismo pago.",
                           "Me aplicaron un cargo por un servicio que jamás solicité.",
                           "La comisión que me facturaron no aparece en lo que firmé.",
                           "Me siguen facturando un producto que di de baja hace meses.",
                           "Pagué a tiempo y aun así me sumaron recargos por atraso.",
                           "Devolví un producto en la tienda y la plata nunca volvió a mi tarjeta."],
        "App y canales digitales": ["No logro ingresar a la aplicación, me rechaza la contraseña.",
                                    "El sitio no carga y no puedo hacer nada en línea.",
                                    "Desde la última actualización la aplicación se cierra sola.",
                                    "Nunca me llega el SMS con el código para autorizar.",
                                    "Quise mandarle plata a un amigo y la aplicación falló.",
                                    "El asistente virtual no entiende nada de lo que le escribo."],
        "Crédito y tasas": ["Quiero pagar por adelantado mi préstamo y la penalización es altísima.",
                            "El interés que me cargan es mayor al pactado.",
                            "Perdí mi empleo y no puedo seguir pagando las cuotas del préstamo.",
                            "No me aprobaron el préstamo para la casa y no me dicen por qué.",
                            "El dividendo de este mes vino más caro sin aviso.",
                            "Pedí aplazar unas cuotas del préstamo y no hay respuesta."],
        "Atención al cliente": ["Nadie contesta el teléfono, estuve 50 minutos en espera.",
                                "La persona que me atendió en la oficina fue muy grosera.",
                                "Me tienen dando vueltas de un departamento a otro.",
                                "Quedaron de llamarme y nunca lo hicieron.",
                                "Me dieron una respuesta incorrecta y ahora estoy peor.",
                                "Llevo semanas esperando que alguien me responda el caso."],
        "Información y trámites": ["¿Qué papeles necesito para abrirle una cuenta a mi hija?",
                                   "Necesito un documento que acredite lo que debo.",
                                   "¿Cómo doy de baja mi cuenta?",
                                   "Quiero que me suban el límite de la tarjeta.",
                                   "¿Dónde saco los movimientos del último semestre?",
                                   "Me piden el comprobante de intereses para el SII."],
    },
    "mercasur": {
        "Despacho": ["El pedido se demoró una eternidad en llegar.", "El courier dejó el paquete en otra dirección.",
                     "Recibí el paquete aplastado y con el contenido dañado.", "Me lo trajeron al día siguiente, rapidísimo.",
                     "Postergaron la entrega varias veces sin avisar.", "El tracking dice que lo recibí pero no tengo nada."],
        "Calidad del producto": ["Se echó a perder a la primera semana de uso.", "Es de plástico barato, se siente muy endeble.",
                                 "Lo que me llegó no tiene nada que ver con las imágenes.", "Funciona de maravilla, muy bien hecho.",
                                 "A los pocos usos empezó a fallar.", "Las costuras se soltaron con el primer lavado."],
        "Precio": ["Me salió más caro que en la competencia.", "Para lo que cuesta, no vale la pena.",
                   "Una ganga, lo encontré a mitad de valor.", "Inflaron el valor justo antes del Black Friday.",
                   "El monto final no era el que mostraba la publicación.", "Excelente oferta, pagué muy poco."],
        "Atención": ["La ejecutiva del chat fue amable y resolvió mi duda.", "Nadie me contesta los mensajes, puros robots.",
                     "El vendedor de la tienda no tenía idea del producto.", "Me tuvieron horas en espera por teléfono.",
                     "Respondieron súper rápido mi consulta por correo.", "Me dieron respuestas genéricas que no resolvieron nada."],
        "Devoluciones y garantía": ["Todavía no me devuelven la plata del producto que retorné.",
                                    "Se negaron a cambiarme el producto defectuoso aunque estaba en garantía.",
                                    "Me cambiaron el producto sin hacer preguntas.",
                                    "El trámite para retornar el producto es una pesadilla.",
                                    "Me reintegraron el dinero rapidísimo.",
                                    "Me exigieron la caja original para aceptar el cambio."],
        "Pago": ["Me descontaron dos veces el mismo monto.", "La tarjeta fue rechazada pero el cargo aparece igual.",
                 "El cupón no se aplicó al finalizar la compra.", "No me dejaron pagar en cuotas.",
                 "El checkout se colgó justo al confirmar la compra.", "Pagar con débito fue fácil y rápido."],
    },
}[CASO]

pa_textos = [t for c, ts in PARAFRASIS.items() for t in ts]
pa_y = np.array([c for c, ts in PARAFRASIS.items() for t in ts])
pa_E = model.encode(pa_textos, normalize_embeddings=True)
print(f"Paráfrasis: {len(pa_textos)} textos · {len(PARAFRASIS)} clases")

In [ ]:
f1 = lambda real, pred: f1_score(real, pred, average="macro")
filas = []
for n in [25, 50, 200, 1000]:
    for semilla in range(3):
        sub = np.random.default_rng(semilla).choice(idx_tr, n, replace=False)
        me = LogisticRegression(C=10, max_iter=2000).fit(E[sub], y[sub])
        tv = TfidfVectorizer(sublinear_tf=True)
        mt = LogisticRegression(C=10, max_iter=2000).fit(tv.fit_transform(textos_np[sub]), y[sub])
        filas.append({"n": n, "modelo": "Embeddings",
                      "Test del caso": f1(y[idx_te], me.predict(E[idx_te])), "Paráfrasis": f1(pa_y, me.predict(pa_E))})
        filas.append({"n": n, "modelo": "TF-IDF",
                      "Test del caso": f1(y[idx_te], mt.predict(tv.transform(textos_np[idx_te]))),
                      "Paráfrasis": f1(pa_y, mt.predict(tv.transform(pa_textos)))})

# Referencia: los modelos entrenados con TODO el train (sección 5)
todo = {"Embeddings": m_emb, "TF-IDF": m_tf}
for nombre, m in todo.items():
    Xte = E[idx_te] if nombre == "Embeddings" else tfidf.transform(textos_np[idx_te])
    Xpa = pa_E if nombre == "Embeddings" else tfidf.transform(pa_textos)
    filas.append({"n": len(idx_tr), "modelo": nombre, "Test del caso": f1(y[idx_te], m.predict(Xte)),
                  "Paráfrasis": f1(pa_y, m.predict(Xpa))})

curva = pd.DataFrame(filas).groupby(["n", "modelo"]).mean().unstack().round(3)
display(curva)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, conjunto in zip(axes, ["Test del caso", "Paráfrasis"]):
    for modelo, estilo in [("Embeddings", "o-"), ("TF-IDF", "s--")]:
        ax.plot(curva.index, curva[(conjunto, modelo)], estilo, label=modelo)
    ax.set_xscale("log"); ax.set_xticks(curva.index, curva.index)
    ax.set_title(conjunto); ax.set_xlabel("ejemplos etiquetados (escala log)"); ax.grid(alpha=0.3)
axes[0].set_ylabel("F1 macro"); axes[0].legend()
plt.suptitle(f"Curva de aprendizaje · {CASO}"); plt.tight_layout(); plt.show()

**Cómo leerlo (y escribirlo en tu informe):**

- En el **test del caso**, TF-IDF es un rival durísimo: los textos de nuestros casos repiten mucho las mismas frases. Con muy pocos datos suelen empatar; con miles, empatan.
- En las **paráfrasis**, los embeddings ganan por 15–20 puntos en todos los tamaños: **generalizan el significado** en vez de memorizar palabras.
- La recomendación de negocio depende de **cuántas etiquetas tienes** y de **qué tan estable es el lenguaje** de tus clientes. Los embeddings no ganan "por magia".

## 7. Mapa 2D de los significados

Los embeddings tienen 384 dimensiones; para mirarlos los proyectamos a 2 con **PCA** (la técnica de reducción de dimensiones: busca las 2 direcciones con más variación). Es una "sombra" del espacio real: úsala para explorar, no para decidir.

In [ ]:
from sklearn.decomposition import PCA

rng = np.random.default_rng(SEMILLA)
muestra = rng.choice(len(df), size=min(2000, len(df)), replace=False)
pca = PCA(n_components=2, random_state=SEMILLA)
Z = pca.fit_transform(E[muestra])
print(f"Varianza que conserva el mapa 2D: {pca.explained_variance_ratio_.sum():.1%}")

fig, ax = plt.subplots(figsize=(9, 7))
for clase in cfg["clases"]:
    m = y[muestra] == clase
    ax.scatter(Z[m, 0], Z[m, 1], s=10, alpha=0.6, label=clase)
ax.set_xlabel("componente principal 1"); ax.set_ylabel("componente principal 2")
ax.set_title(f"Mapa 2D de los embeddings ({CASO}) · coloreado por clase")
ax.legend(markerscale=2, fontsize=8); plt.tight_layout(); plt.show()

**¿Qué mirar?** ¿Qué clases forman "islas" y cuáles se mezclan? Las que se mezclan en el mapa suelen ser las que más se confunden en la matriz de confusión de la sección 5. ¿Tiene sentido para el negocio?

## 8. Descubrir temas sin etiquetas (K-means)

Ahora **olvidamos las etiquetas** y dejamos que K-means agrupe los textos por cercanía. Para elegir **k** miramos la **silueta** (de −1 a 1; más alto = grupos más compactos y separados). En texto suele ser baja y plana: úsala como guía y decide también por **interpretabilidad**.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

Em = E[muestra]          # usamos la muestra de 2.000 para que sea rápido
siluetas = {}
for k in range(2, 13):
    etiquetas_k = KMeans(n_clusters=k, n_init=5, random_state=SEMILLA).fit_predict(Em)
    siluetas[k] = silhouette_score(Em, etiquetas_k, metric="cosine")

plt.figure(figsize=(6, 3.5))
plt.plot(list(siluetas), list(siluetas.values()), "o-")
plt.xlabel("k"); plt.ylabel("silueta (coseno)"); plt.title("¿Cuántos temas?"); plt.grid(alpha=0.3); plt.show()
pd.Series(siluetas, name="silueta").round(3).to_frame().T

Elige `K` mirando el gráfico **y** pensando en el negocio (¿cuántos temas puede gestionar un equipo?). Luego agrupamos **todo** el dataset y le ponemos nombre a cada cluster con sus **palabras distintivas**: las que son más frecuentes en el cluster que en el resto (una versión simple de lo que hace la librería BERTopic).

In [ ]:
K = 8   # 👈 cambia según tu gráfico de silueta y tu criterio de negocio

kmeans = KMeans(n_clusters=K, n_init=10, random_state=SEMILLA)
cluster = kmeans.fit_predict(E)

VACIAS = """a al algo como cómo con de del el ella en es esta este esto fue ha hace hay he hola la las le lo los me mi
mis muy más no nos o para pero por que qué se si sin su sus te un una uno y ya yo buenas buenos días tardes estimados
señores fintrust mercasur saludos atento quedo espero pronta respuesta necesito ayuda urgente escribo segunda vez
dejar reclamo formal muchas gracias antemano ojalá ayuden vergüenza voy ir sernac solucionan cambio banco
compré compra experiencia cuento recomendado vuelvo volvería comprar estrellas ojo tienda bien general decepcionado
feliz""".split()

cv = TfidfVectorizer(stop_words=VACIAS, token_pattern=r"(?u)\b[a-záéíóúñ]{3,}\b", min_df=5)
W = cv.fit_transform(df[COL_TEXTO].str.lower())
vocab = np.array(cv.get_feature_names_out())
media_global = np.asarray(W.mean(axis=0)).ravel()

resumen = []
for c in range(K):
    en_c = cluster == c
    media_c = np.asarray(W[en_c].mean(axis=0)).ravel()
    distintivas = vocab[np.argsort(-(media_c - media_global))[:6]]
    resumen.append({"cluster": c, "n": int(en_c.sum()), "palabras distintivas": ", ".join(distintivas),
                    "clase más común": pd.Series(y[en_c]).mode()[0],
                    "% de esa clase": round(100 * (pd.Series(y[en_c]) == pd.Series(y[en_c]).mode()[0]).mean())})
resumen = pd.DataFrame(resumen).set_index("cluster")
resumen

In [ ]:
# Lee 3 ejemplos de cada cluster antes de ponerle nombre
for c in range(K):
    print(f"\n=== Cluster {c} · {resumen.loc[c, 'palabras distintivas']}")
    for t in df[COL_TEXTO][cluster == c].sample(3, random_state=SEMILLA):
        print("   ·", t[:130])

✍️ **Ponle nombre a cada cluster** (completa el diccionario). Un buen nombre es **de negocio** ("Cobros duplicados de compras"), no técnico ("cluster 3").

⚠️ **Ojo con el estilo:** el embedding resume *todo* el texto, no solo el tema. Si ves un cluster con palabras sin relación entre sí, revisa sus ejemplos: puede estar agrupando los textos escritos **EN MAYÚSCULAS** o con un saludo/firma común. Es un hallazgo válido para tu informe (y una razón para normalizar el texto).

🛍️ **En MercaSur** es probable que aparezcan clusters por **tipo de producto** (por ejemplo, "secador, plancha, crema": belleza) en vez de por **aspecto** (despacho, precio…). No es un error: el embedding agrupa por lo que más "pesa" en el texto, y el producto pesa mucho. Si tu pregunta de negocio es por aspecto, esto muestra por qué el clustering **no reemplaza** a un clasificador entrenado con etiquetas.

In [ ]:
NOMBRES = {c: f"Tema {c}" for c in range(K)}   # 👈 reemplaza por nombres de negocio, p. ej. {0: "Fallas de la app al transferir", ...}

df["cluster"] = cluster
df["tema"] = df["cluster"].map(NOMBRES)
print("Proporción de textos en MAYÚSCULAS por cluster:")
display(df.assign(mayus=df[COL_TEXTO].str.isupper()).groupby("tema")["mayus"].mean().round(2).to_frame().T)
pd.crosstab(df["tema"], df[COL_CLASE])

## 9. Lo que guardaste para la Clase 4

En la carpeta del notebook quedaron:

| Archivo | Contenido | Para qué |
|---|---|---|
| `embeddings_{CASO}.npy` | Matriz de embeddings (una fila por texto, 384 columnas, normalizados) | Reutilizarlos sin recalcular (Clase 4 · RAG y Sprint 3) |
| `textos_limpios_{CASO}.csv` | Texto y clase en el **mismo orden** que la matriz | Saber a qué texto corresponde cada fila |
| `embeddings_{CASO}_modelo.txt` | Nombre del modelo | Nunca mezclar vectores de modelos distintos |

⚠️ Colab borra los archivos al cerrar la sesión: descárgalos o cópialos a tu Drive (`from google.colab import drive; drive.mount("/content/drive")`).

## ✍️ Ejercicios para tu proyecto (Sprint 2 · domingo 15 de noviembre, 23:59)

1. **Embeddings de tu caso:** reporta el modelo, el número de textos y el tiempo de cálculo (sección 3). Guarda el `.npy`.
2. **Clasificador sobre embeddings vs. baselines:** agrega a la tabla de la sección 5 tu **MLP del Sprint 1** (puedes entrenar la misma MLP de Keras usando `E` como entrada en vez de TF-IDF). Reporta F1 macro y matriz de confusión en el **mismo test**.
3. **Curva con pocos datos:** presenta la curva 50 / 200 / 1.000 de la sección 6. ¿A partir de cuántos ejemplos TF-IDF alcanza a los embeddings en tu caso? ¿Qué pasa con las paráfrasis? Escribe **6 paráfrasis propias** (1 por clase) y agrégalas a la prueba.
4. **Buscador semántico:** reemplaza `CONSULTAS` por **5 consultas de negocio tuyas** (piensa en quién las haría: el área de riesgo, operaciones, marketing…). Para cada una, cuenta cuántos de los 5 resultados son relevantes, y compara con `buscar_tfidf` en al menos dos.
5. **Mapa de temas:** justifica tu `K` (silueta + interpretación), completa `NOMBRES` con nombres de negocio y muestra 2–3 ejemplos por cluster. ¿Encontraste algún tema que **no** coincide con las categorías oficiales?
6. **Desafío (opcional):** normaliza los textos escritos completamente en mayúsculas (`texto.lower()` si `texto.isupper()`), recalcula los embeddings y compara el F1 y los clusters. ¿Mejora?
7. **Conclusión de negocio (1 párrafo):** ¿recomendarías embeddings o TF-IDF para tu caso? Usa tus números y considera costo, robustez ante textos nuevos y facilidad de explicación.